# 01. Dựng dataset

Sinh KB tổng hợp tiếng Việt (2 tenant), xuất mẫu kiểm tra nhãn, tải ViNLI. Phần Jira là tuỳ chọn.

In [10]:
import os, sys, subprocess
from importlib import metadata
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
    core = ("numpy", "pandas", "scipy", "scikit-learn", "pyarrow")
    def installed_versions():
        found = {}
        for name in core:
            try:
                found[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                found[name] = None
        return found
    before = installed_versions()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    if installed_versions() != before:
        print("Đã cài phiên bản mới của numpy/pandas/scipy. Runtime tự khởi động lại, đợi vài giây rồi chạy lại notebook từ đầu.")
        os.kill(os.getpid(), 9)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

Thư mục làm việc: /content/repo


In [11]:
# Tuỳ chọn: giữ data/ và artifacts/ trên Google Drive giữa các session Colab
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/conflict-detection"
    for folder in ("data/processed", "artifacts"):
        os.makedirs(f"{target}/{folder}", exist_ok=True)
        subprocess.run(f"cp -rn {folder}/. {target}/{folder}/ && rm -rf {folder} && ln -s {target}/{folder} {folder}", shell=True, check=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## KB tổng hợp

In [12]:
from src import pipeline
from src.io_utils import read_parquet
from src.paths import DATA_PROCESSED
pipeline.generate_kb()
docs = read_parquet(DATA_PROCESSED / "kb_docs.parquet")
pairs = read_parquet(DATA_PROCESSED / "kb_pairs.parquet")
edges = read_parquet(DATA_PROCESSED / "kb_edges.parquet")
print(len(docs), "tài liệu,", len(pairs), "cặp,", int(pairs.label.sum()), "cặp mâu thuẫn")
pairs.groupby(["tenant", "variant_type", "label"]).size().unstack("tenant")

972 tài liệu, 1430 cặp, 286 cặp mâu thuẫn


,tenant,policy,tech
variant_type,label,,
conflict,1,81,78
hidden_conflict,1,61,66
lexical_overlap,0,57,64
paraphrase,0,83,82
random,0,300,300
same_template,0,129,129


In [13]:
example = pairs[pairs.variant_type == "conflict"].iloc[0]
text = docs.set_index("doc_id")["text"]
print(text[example.doc_a]); print("-" * 60); print(text[example.doc_b])

SOP-008 – Chính sách cấp quyền truy cập – Hệ thống báo cáo
Phiên bản: 1 | Ngày ban hành: 06/11/2022 | Đơn vị ban hành: Bảo mật
Điều 1. Quyền truy cập vào hệ thống báo cáo được rà soát lại mỗi 12 tháng.
Điều 2. Thời hạn thu hồi quyền khi nhân viên nghỉ việc là 4 giờ.
Điều 3. Độ dài mật khẩu không được ít hơn 8 ký tự.
Điều 4. Tài khoản quản trị phải thay mật khẩu với chu kỳ 180 ngày.
Điều 5. Tài khoản bị khoá sau 5 lần đăng nhập sai liên tiếp.
Tài liệu liên quan: SOP-133, SOP-090.
------------------------------------------------------------
SOP-008 – Chính sách cấp quyền truy cập – Hệ thống báo cáo
Phiên bản: 2 | Ngày ban hành: 04/04/2023 | Đơn vị ban hành: Bảo mật
Cập nhật từ SOP-008 phiên bản 1.
Điều 1. Quyền truy cập vào hệ thống báo cáo được rà soát lại mỗi 1 tháng.
Điều 2. Quyền của nhân viên nghỉ việc phải bị thu hồi trong vòng 4 giờ.
Điều 3. Mật khẩu phải có tối thiểu 8 ký tự.
Điều 4. Mật khẩu tài khoản quản trị được đổi mỗi 180 ngày.
Điều 5. Sau 5 lần nhập sai mật khẩu, tài khoản

## Mẫu 100 cặp để nhóm gán nhãn lại

Mỗi người thêm một cột `annot_<tên>` (1 = mâu thuẫn, 0 = không), rồi chạy ô tính kappa.

In [14]:
from src.synth.qa_sample import export_blind_sample
sample = export_blind_sample(docs, pairs, n=100, seed=42)
sample.to_csv(DATA_PROCESSED / "kb_qa_sample.csv", index=False)
sample.head()

,pair_id,doc_a,doc_b,text_a,text_b
0,1,SOP-118-v1,SOP-047-v1,SOP-118 – Quy trình xử lý sự cố – Hệ thống ema...,SOP-047 – Quy trình xử lý sự cố – Dịch vụ gửi ...
1,2,POL-134-v1,POL-197-v1,POL-134 – Quy trình giao nhận đơn hàng – Khối ...,POL-197 – Quy trình phê duyệt đơn hàng mua sắm...
2,3,POL-162-v1,POL-107-v1,POL-162 – Quy định công tác phí – Chi nhánh Lo...,POL-107 – Quy định nghỉ phép – Chi nhánh Thái ...
3,4,SOP-121-v1,SOP-104-v1,SOP-121 – Chính sách cấp quyền truy cập – Hệ t...,SOP-104 – Chính sách cấp quyền truy cập – Hệ t...
4,5,SOP-153-v1,SOP-295-v1,SOP-153 – Chính sách cấp quyền truy cập – Nền ...,SOP-295 – Chính sách cấp quyền truy cập – Nền ...


In [15]:
from pathlib import Path
import pandas as pd
from src.synth.qa_sample import kappa_report, disagreements
annotated_path = DATA_PROCESSED / "kb_qa_sample_annotated.csv"
if annotated_path.exists():
    annotated = pd.read_csv(annotated_path)
    display(kappa_report(annotated, pairs))
    display(disagreements(annotated, pairs))
else:
    print("Chưa có file gán nhãn:", annotated_path)

Chưa có file gán nhãn: /content/repo/data/processed/kb_qa_sample_annotated.csv


## ViNLI

In [16]:
from src.datasets.vinli import load_vinli
vinli_test = load_vinli("test")
vinli_test.label.value_counts()

label
neutral          777
entailment       750
contradiction    737
Name: count, dtype: int64

## Jira (tuỳ chọn)

Dữ liệu thật từ Public Jira Dataset (Montgomery và cộng sự, MSR 2022; Zenodo doi 10.5281/zenodo.5882881). Đặt `RUN_JIRA = True` rồi chạy các ô bên dưới. Notebook tự làm các bước sau:

1. Cài MongoDB 8.0 và `mongorestore` qua apt (Colab chạy Ubuntu, có quyền root).
2. Tải file zip khoảng 5,8 GB, tự tải tiếp nếu bị ngắt, rồi kiểm tra MD5.
3. Chỉ restore các collection trong `JIRA_COLLECTIONS`. Archive được đọc thẳng từ zip, không giải nén. Restore toàn bộ dump cần khoảng 60 GB, vượt dung lượng đĩa Colab.
4. Thống kê số issue và số link nhãn (Duplicate, Supersede, Replace) theo project. Nếu `JIRA_PROJECTS = None` thì chọn `JIRA_PROJECT_COUNT` project có nhiều link nhãn nhất trong khoảng `JIRA_PROJECT_SIZE` issue. Mỗi project là một tenant, nên Layer 2 chọn λ và τ riêng cho từng project.

Mặc định là collection `Spring` với 3 project ROO, SEC, DATAMONGO (khoảng 9.700 issue). Đổi `JIRA_COLLECTIONS` nếu muốn instance khác, ví dụ `Hyperledger`, `MongoDB`, `Sakai`.

Lần đầu mất khoảng 30–40 phút, chủ yếu là tải và đọc hết archive. Khi `USE_DRIVE = True`, file zip được tải thẳng vào `MyDrive/conflict-detection/jira/`, và `data/processed/jira_*.parquet` cũng nằm trên Drive. Các session sau dùng lại hai thứ này, không phải tải lại. Nếu bạn đã có sẵn file zip, chỉ cần đặt nó vào thư mục đó và giữ nguyên tên `2025-06-23 ThePublicJiraDataset.zip`, hoặc trỏ `JIRA_DUMP_DIR` tới thư mục chứa file.

In [ ]:
from pathlib import Path
RUN_JIRA = True
JIRA_COLLECTIONS = ["Spring"]
JIRA_COLLECTION = "Spring"
JIRA_PROJECTS = None
JIRA_PROJECT_COUNT = 3
JIRA_PROJECT_SIZE = (500, 5000)
JIRA_WORKDIR = Path("/content/jira") if IN_COLAB else Path("data/raw/jira")
JIRA_DUMP_DIR = Path("/content/drive/MyDrive/conflict-detection/jira") if IN_COLAB and USE_DRIVE else JIRA_WORKDIR
JIRA_REUSE_PARQUET = True

In [ ]:
jira_ready = JIRA_REUSE_PARQUET and (DATA_PROCESSED / "jira_docs.parquet").exists()
if RUN_JIRA and not jira_ready:
    from src.datasets import jira_setup
    jira_setup.install_mongodb()
    zip_path = jira_setup.download_dump(JIRA_DUMP_DIR)
    mongo_uri = jira_setup.start_mongod(JIRA_WORKDIR / "mongo-data")
    print(jira_setup.restore_collections(zip_path, JIRA_COLLECTIONS, uri=mongo_uri))
    overview = jira_setup.project_overview(JIRA_COLLECTION, uri=mongo_uri)
    display(overview.head(20))
    jira_projects = JIRA_PROJECTS or jira_setup.suggest_projects(
        overview, count=JIRA_PROJECT_COUNT, min_issues=JIRA_PROJECT_SIZE[0], max_issues=JIRA_PROJECT_SIZE[1])
    print("Project được chọn:", jira_projects)

In [ ]:
if RUN_JIRA:
    from src.datasets.jira_link_stats import link_type_counts, resolution_counts, project_summary
    from src.io_utils import write_parquet
    if jira_ready:
        jira_docs = read_parquet(DATA_PROCESSED / "jira_docs.parquet")
        jira_edges = read_parquet(DATA_PROCESSED / "jira_edges.parquet")
    else:
        from src.datasets.jira_export import export_project
        jira_docs, jira_edges = export_project(mongo_uri, jira_setup.DATABASE, JIRA_COLLECTION, project_key=jira_projects)
        write_parquet(jira_docs, DATA_PROCESSED / "jira_docs.parquet")
        write_parquet(jira_edges, DATA_PROCESSED / "jira_edges.parquet")
    display(project_summary(jira_docs, jira_edges))
    display(link_type_counts(jira_edges).head(20))
    display(resolution_counts(jira_docs))

In [ ]:
if RUN_JIRA:
    from src.datasets.jira_pairs import build_pairs
    pipeline.embed("jira", backend="bge-m3")
    neighbors = read_parquet(pipeline.artifact_dir("jira") / "neighbors.parquet")
    jira_pairs = build_pairs(jira_docs, jira_edges, neighbors, seed=42)
    write_parquet(jira_pairs, DATA_PROCESSED / "jira_pairs.parquet")
    display(jira_pairs.groupby(["variant_type", "label"]).size())